In [0]:
from pyspark.sql.functions import to_timestamp, col, date_trunc, hour, expr, regexp_replace, try_to_timestamp
from graphframes import GraphFrame
from pyspark.sql.functions import to_date

In [0]:
1

In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041026_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041026_otpw_test_df.parquet")

In [0]:
def get_shape(df):
  '''Return the shape of the dataframe'''
  return f"{df.count()} rows, {len(df.columns)} columns"


In [0]:
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
display(dbutils.fs.ls("dbfs:/mnt/mids-w261"))

In [0]:
otpw_df = spark.read.parquet(f"dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
otpw_df.printSchema()

In [0]:
get_shape(otpw_df)

In [0]:
display(otpw_df)

# Build nodes and edges, then build the graph

In [0]:

%skip
# Airports are nodes, flights are edges
nodes = otpw_df.select(col("ORIGIN").alias("id")).distinct()

# Edges: flights
edges = otpw_df.select(
    col("ORIGIN").alias("src"),
    col("DEST").alias("dst")
)

# Build the graph
g = GraphFrame(nodes, edges)

# Run PageRank 

In [0]:
%skip
pr = g.pageRank(resetProbability=0.15, maxIter=20)
pagerank_df = pr.vertices.select("id", "pagerank")

# Join PageRank back to OTPW
otpw_with_pr = otpw_df.join(
    pagerank_df,
    otpw_df.ORIGIN == pagerank_df.id,
    "left"
).drop("id")

Every flight should now have a PageRank‑based feature representing the structural importance of its origin airport.

In [0]:
%skip
display(otpw_with_pr)

In [0]:
%skip
display(otpw_with_pr.select('ORIGIN', 'ORIGIN_CITY_NAME', 'pagerank').distinct())

# Temporal PageRank by Month

In [0]:
otpw_df = otpw_df.withColumn("flight_date", to_date("FL_DATE"))

# Compute PageRank per month
months = otpw_df.select("YEAR", "MONTH").distinct().collect()

pagerank_monthly = []

for row in months:
    y, m = row["YEAR"], row["MONTH"]
    df_month = otpw_df.filter((col("YEAR") == y) & (col("MONTH") == m))

    vertices = df_month.select(col("ORIGIN").alias("id")).distinct()
    edges = df_month.select(col("ORIGIN").alias("src"), col("DEST").alias("dst"))

    g = GraphFrame(vertices, edges)
    pr = g.pageRank(resetProbability=0.15, maxIter=20)

    pr_month = pr.vertices.withColumn("YEAR", lit(y)).withColumn("MONTH", lit(m))
    pagerank_monthly.append(pr_month)

pagerank_monthly_df = reduce(DataFrame.unionByName, pagerank_monthly)